<a href="https://colab.research.google.com/github/awaiskhan005/Agentic-AI/blob/main/AI_AGENT_prototype_for_XACT_accounting_and_financial_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# To automate accounting processes using AI agents, I can provide a simplified prototype using a multi-agent system. Here, I'll demonstrate a basic prototype that automates data processing, invoice matching, and credit control in an accounting context. We'll focus on using some real data like invoices and bank statements, simulating agents that automate different steps of the process.

Steps for the Prototype:
Data Collection: Scrape invoices, bank statements, or any other financial data.
Data Processing: Automatically categorize transactions, match invoices with payments, and process them.
Credit Control: Automatically send reminders or alerts for overdue payments.
## Automation Tools: We’ll use crewAI to simulate the multi-agent collaboration, with tools to handle data scraping and processing.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [ ]:
from crewai import Agent, Task, Crew

**Note**:
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [ ]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [ ]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

## Creating Tasks

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

- Display the final result as Markdown.

In [ ]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 172.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.9/507.9 kB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 82.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.1.2 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
import pandas as pd

# Sample Invoice and Bank Data (Real Data Simulation)
invoice_data = {
    'Invoice Number': ['INV001', 'INV002', 'INV003'],
    'Amount Due': [500, 300, 1000],
    'Due Date': ['2025-03-01', '2025-02-28', '2025-03-10'],
    'Client': ['Client A', 'Client B', 'Client C']
}

bank_data = {
    'Transaction ID': ['TX001', 'TX002', 'TX003'],
    'Amount Paid': [500, 300, 1000],
    'Payment Date': ['2025-02-28', '2025-02-28', '2025-03-01'],
    'Client': ['Client A', 'Client B', 'Client C']
}

# Convert the data to DataFrames for easier manipulation
invoices_df = pd.DataFrame(invoice_data)
bank_df = pd.DataFrame(bank_data)

# Agent for Invoice Scraping (simulating scraping from a website)
invoice_scraper_agent = Agent(
    role="Invoice Scraper",
    goal="Scrape invoices from a website and categorize them",
    backstory="This agent scrapes invoice data from a website and categorizes them into a structured format for further processing.",
    tools=[ScrapeWebsiteTool()],
    verbose=True
)

# Agent for Payment Matching (matches bank payments with invoices)
payment_matching_agent = Agent(
    role="Payment Matcher",
    goal="Match payments from bank with invoices",
    backstory="This agent matches payments recorded in the bank transactions to outstanding invoices, ensuring accurate financial records.",
    verbose=True
)

# Task to scrape invoices
scrape_invoices_task = Task(
    description="Scrape and extract invoices data from the website.",
    expected_output="Extracted invoice data.",
    agent=invoice_scraper_agent
)

# Task to match payments with invoices
match_payments_task = Task(
    description="Match payments made in the bank statement with invoices.",
    expected_output="Payment matched to the correct invoice.",
    agent=payment_matching_agent
)

# Creating the Crew (multi-agent collaboration)
financial_automation_crew = Crew(
    agents=[invoice_scraper_agent, payment_matching_agent],
    tasks=[scrape_invoices_task, match_payments_task],
    verbose=True
)

# Simulate scraping invoices (In practice, you would scrape a website for data)
result = financial_automation_crew.kickoff(inputs={'invoices': invoices_df, 'bank_data': bank_df})

# Displaying the results
matched_payments = pd.merge(invoices_df, bank_df, on='Client', how='inner')
print("Matched Payments:")
print(matched_payments)

# Automating Credit Control (send reminders for unpaid invoices)
unpaid_invoices = invoices_df[~invoices_df['Invoice Number'].isin(matched_payments['Invoice Number'])]
if not unpaid_invoices.empty:
    for index, row in unpaid_invoices.iterrows():
        print(f"Reminder: {row['Client']} - Invoice {row['Invoice Number']} is overdue by {row['Due Date']}.")


 [DEBUG]: == Working Agent: Invoice Scraper
 [INFO]: == Starting Task: Scrape and extract invoices data from the website.


> Entering new CrewAgentExecutor chain...
I need to start by reading the website content to extract the invoice data.

Action: Read website content
Action Input: {"website_url": "https://examplewebsite.com/invoices"} 

Page not found – Example Website
Skip to content 
Example Website
Menu 
Menu 
Home
Blog
About
Contact
Oops! That page can’t be found.
It looks like nothing was found at this location. Maybe try searching?
Search for:
Search
RECENT POSTS
By 
Footer Widget 1
This is just an example text widget. You can add any kind of widget in here.
Mauris lacinia rhoncus lacus, eget bibendum ex gravida ac. Proin sed mauris ipsum. Sed eu commodo ipsum.
Footer Widget 2
This is just an example text widget. You can add any kind of widget in here.
Mauris lacinia rhoncus lacus, eget bibendum ex gravida ac. Proin sed mauris ipsum. Sed eu commodo ipsum.
Facebook
Twitter
Lin

In [ ]:
pip install numpy

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.1.2 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
